# 🧠 Aula 10 — Introdução ao ROCm e GPUs AMD

**Objetivo:** configurar e executar aplicações de IA em GPUs AMD usando o ecossistema ROCm,
compreender a portabilidade via HIP e rodar modelos PyTorch em ambiente Docker, promovendo
independência de fabricante no desenvolvimento de IA.

**Roteiro deste notebook:**
1. Diagnóstico do ambiente (CUDA, ROCm/HIP ou CPU).
2. Teoria: a stack ROCm e a equivalência com CUDA.
3. Portabilidade: o mesmo código Python, backends diferentes.
4. Atividade: benchmark de matmul e treino (ResNet-18/CNN).
5. Docker: a imagem `rocm/pytorch`.
6. Discussão e síntese.

> 💡 **Sem GPU AMD?** No Colab (NVIDIA/CUDA) o mesmo notebook roda e detecta `CUDA`. Numa
> máquina sem GPU, ele usa a CPU e mostra a referência — a aula roda do começo ao fim.

## 1. Diagnóstico do ambiente

O PyTorch esconde a diferença entre NVIDIA e AMD: `torch.cuda.is_available()` é `True` nos
dois casos. O que distingue é `torch.version.hip` (ROCm) vs. `torch.version.cuda` (NVIDIA).

In [ ]:
# @title 🔍 Diagnóstico: CUDA, ROCm/HIP ou CPU?
# ============================================================================
# OBJETIVO: descobrir qual backend o PyTorch está usando neste ambiente.
# ============================================================================
import platform

print(f"Sistema : {platform.system()} {platform.release()}")
print(f"Python  : {platform.python_version()}")

try:
    import torch
    print(f"PyTorch : {torch.__version__}")
    if torch.cuda.is_available():
        print(f"GPU     : {torch.cuda.get_device_name(0)}")
        if getattr(torch.version, 'hip', None):
            print(f"Backend : ROCm / HIP (AMD) - {torch.version.hip}")
        else:
            print(f"Backend : CUDA Nativo (NVIDIA) - {torch.version.cuda}")
    else:
        print("GPU     : NENHUMA (executando em CPU)")
except ImportError:
    print("PyTorch não instalado. No Colab:  !pip install torch -q")

## 2. A stack ROCm e a equivalência com CUDA

**ROCm** (Radeon Open Compute) é a plataforma open-source da AMD. A camada **HIP** permite
rodar código escrito para CUDA com **mínimas mudanças** (e, em PyTorch, nenhuma).

| CUDA (NVIDIA) | ROCm (AMD) |
| :--- | :--- |
| `nvcc` | `hipcc` |
| cuBLAS | rocBLAS |
| cuDNN | MIOpen |
| cuFFT | rocFFT |
| nvidia-smi | rocm-smi |
| nvidia/cuda | rocm/pytorch |

A tabela completa de equivalentes está em `laboratorio_windows/2_diagnostico_portabilidade.py`.

In [ ]:
# @title 🔁 Tabela de equivalência CUDA x ROCm
# ============================================================================
# OBJETIVO: visualizar o mapeamento entre os dois ecossistemas.
# ============================================================================
equivalentes = [
    ("Compilador",        "nvcc",          "hipcc"),
    ("Álgebra linear",    "cuBLAS",        "rocBLAS"),
    ("Deep learning",     "cuDNN",         "MIOpen"),
    ("FFT",               "cuFFT",         "rocFFT"),
    ("Monitoramento",     "nvidia-smi",    "rocm-smi"),
    ("Container",         "nvidia/cuda",   "rocm/pytorch"),
]
print(f"{'Componente':<18} {'CUDA':<16} {'ROCm'}")
print("-" * 50)
for nome, cuda_eq, rocm_eq in equivalentes:
    print(f"{nome:<18} {cuda_eq:<16} {rocm_eq}")

## 3. Portabilidade: o mesmo código, backends diferentes

Escrevemos o código PyTorch normalmente (usando `"cuda"`). Ele roda em NVIDIA **e** AMD,
porque o ROCm emula a API CUDA via HIP. **Zero linhas alteradas.**

In [ ]:
# @title ⏱️ Benchmark portável: matmul GPU vs. CPU
# ============================================================================
# OBJETIVO: medir a multiplicação de matrizes no dispositivo disponível.
# Este código é idêntico em CUDA e ROCm.
# ============================================================================
import time

try:
    import torch
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    N = 2048
    A = torch.randn(N, N, device=dispositivo, dtype=torch.float32)
    B = torch.randn(N, N, device=dispositivo, dtype=torch.float32)

    def medir(device):
        a, b = A.to(device), B.to(device)
        if device == "cuda": torch.cuda.synchronize()
        inicio = time.perf_counter()
        for _ in range(10):
            _ = torch.matmul(a, b)
        if device == "cuda": torch.cuda.synchronize()
        return (time.perf_counter() - inicio) / 10

    t_gpu = medir(dispositivo)
    print(f"MatMul {N}x{N} em {dispositivo.upper()}: {t_gpu*1000:.2f} ms")
    if dispositivo == "cuda":
        t_cpu = medir("cpu")
        print(f"MatMul {N}x{N} na CPU          : {t_cpu*1000:.2f} ms")
        print(f"Speedup GPU/CPU                : {t_cpu/t_gpu:.1f}x")
except ImportError:
    print("PyTorch não instalado — sem benchmark.")

## 4. Atividade: treino sintético (ResNet-18 / CNN)

Treinamos alguns batches e medimos o **throughput** (imagens/segundo). Sem a `torchvision`,
usamos uma CNN pequena com o mesmo formato de entrada — o objetivo é medir o dispositivo.

In [ ]:
# @title 🏋️ Treino sintético e throughput
# ============================================================================
# OBJETIVO: medir throughput de treino no dispositivo disponível.
# ============================================================================
try:
    import torch, torch.nn as nn

    dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Usa ResNet-18 se houver torchvision; senão, uma CNN simples equivalente.
    try:
        import torchvision
        modelo = torchvision.models.resnet18(weights=None)
        rotulo = "ResNet-18"
    except ImportError:
        modelo = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 1000))
        rotulo = "CNN simples (sem torchvision)"

    modelo = modelo.to(dispositivo)
    criterio = nn.CrossEntropyLoss()
    otimizador = torch.optim.SGD(modelo.parameters(), lr=0.01, momentum=0.9)
    modelo.train()

    BATCH, N_BATCHES = 64, 10
    forma = (BATCH, 3, 224, 224)
    print(f"Treinando {rotulo} em {dispositivo.type.upper()}")

    for i in range(N_BATCHES):
        imgs = torch.randn(*forma, device=dispositivo)
        labels = torch.randint(0, 1000, (BATCH,), device=dispositivo)
        inicio = time.perf_counter()
        otimizador.zero_grad()
        perda = criterio(modelo(imgs), labels)
        perda.backward(); otimizador.step()
        if dispositivo.type == "cuda": torch.cuda.synchronize()
        t = time.perf_counter() - inicio
        print(f"  Batch {i+1:02d}/{N_BATCHES} | Perda {perda.item():.4f} | {BATCH/t:.0f} imgs/s")
except ImportError:
    print("PyTorch não instalado — sem treino.")

## 5. Docker: a imagem `rocm/pytorch`

A forma recomendada de usar ROCm é via container — sem instalar drivers no host. O passo a
passo está em `laboratorio_rocm-docker/README.md`.

```bash
docker pull rocm/pytorch:rocm6.2_ubuntu22.04_py3.10_pytorch_release_2.3.0
docker run -it --device=/dev/kfd --device=/dev/dri \
  --group-add=video --group-add=render --ipc=host --shm-size 8G \
  rocm/pytorch:... python3 /workspace/rocm_pytorch_benchmark.py
```

> No Colab **não** há GPU AMD nem `rocm-smi` — este notebook roda com CUDA (ou CPU), mas o
> **código é o mesmo** que rodaria no container ROCm.

## 6. Discussão em Grupo

Em grupos de 3–4, no cenário das GPUs AMD:

1. Se o PyTorch roda CUDA em AMD via ROCm sem modificação, por que o **CUDA ainda domina**?
2. Resultados treinados com cuDNN e migrados para MIOpen são **numericamente idênticos**?
3. Docker facilita o ROCm, mas adiciona overhead. Quando seria **problemático**?
4. H100 (US$30k) vs. MI300X (US$20k): além do preço, que fatores pesam?

> Atividade de pesquisa completa em `aulas/aula10/atividade.md`.

## 7. Exercícios (5)

Resolva os 5 exercícios **neste notebook**. O valor está em **experimentar e explicar**.

---

**1) A stack ROCm.** Desenhe (em texto) a pilha do ROCm, de cima para baixo: Aplicação →
Framework → **HIP** → ROCr → KFD → Hardware. Qual camada garante a portabilidade?

**2) Zero código.** Explique por que `torch.cuda.is_available()` retorna `True` **também**
numa GPU AMD com ROCm. Que impacto isso tem para quem já tem código PyTorch em CUDA?

**3) Diagnóstico.** Na célula-esqueleto, detecte se o backend é CUDA, ROCm/HIP ou CPU e
imprima a conclusão. Rode no Colab (deve detectar CUDA).

**4) Equivalências.** Complete a tabela: `nvcc→?`, `cuBLAS→?`, `cuDNN→?`, `nvidia-smi→?`,
`nvidia/cuda→?`.

**5) Decisão (TCO).** A MI300X custa ~40% menos que a A100. Cite **3 fatores além do preço**
que pesam na decisão de migrar toda uma equipe para AMD.


In [ ]:
# @title Exercício 3 — detectar o backend (CUDA / ROCm / CPU)
# ============================================================================
# OBJETIVO: identificar qual ecossistema o PyTorch está usando.
# Torch.version.hip indica ROCm (AMD); caso contrário, CUDA (NVIDIA) ou CPU.
# ============================================================================
try:
    import torch

    if torch.cuda.is_available():
        if getattr(torch.version, "hip", None):
            backend = f"ROCm/HIP (AMD) {torch.version.hip}"
        else:
            backend = f"CUDA nativo (NVIDIA) {torch.version.cuda}"
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        backend = "CPU (nenhuma GPU acessível)"

    print(f"Backend detectado: {backend}")
    print("O MESMO código Python roda em NVIDIA e AMD sem alterações.")
except ImportError:
    print("PyTorch não instalado. No Colab:  !pip install torch -q")

## 8. Síntese e Tarefa de Casa

**O que levar:**
- **ROCm:** plataforma open-source da AMD — equivalente ao CUDA.
- **HIP:** camada de portabilidade — converte CUDA → AMD com mínimas mudanças.
- **rocBLAS / MIOpen / rocFFT:** equivalentes a cuBLAS / cuDNN / cuFFT.
- **Docker + rocm/pytorch:** forma recomendada — evita conflitos de driver no host.
- **`torch.cuda.is_available()`:** `True` também no ROCm — mesma API.
- **`rocm-smi`:** monitorar GPU AMD (temperatura, VRAM, utilização).

**Tarefa (opcional):** compare o treino da ResNet-50 em CUDA (Colab) vs. ROCm (Docker) e
produza um relatório:
- tempo por época, throughput (imgs/s), VRAM, potência (W);
- trace com `torch.profiler` nos dois ambientes;
- documente o setup (CUDA vs. Docker ROCm);
- calcule o **TCO** para 1 ano de treino.

> 🔗 **Próxima aula:** *Aplicação de Modelos em GPUs NVIDIA e AMD* — vamos além do benchmark:
> medir, registrar com W&B e decidir com dados.